enterprise_pipeline/
│
├── extractors/
│   ├── soap_client.py
│   ├── extract_job_list.py
│   ├── extract_order_details.py
│   ├── extract_invoice_details.py
│
├── loaders/
│   └── snowflake_loader.py
│
├── reporting/
│   ├── order_summary_report.py
│   └── invoice_aging_report.py
│
├── config/
│   └── endpoints.yml
│
├── dbt/
│   └── enterprise_analytics/
│
├── .env
├── requirements.txt
└── main.py

In [1]:
from zeep import Client
from datetime import datetime
import os

In [2]:
import requests

url = "http://128.10.10.3/EnterpriseWebService/Service.asmx"

try:
    r = requests.get(url, timeout=10)
    print(r.status_code)
    print(r.text[:500])
except Exception as e:
    print(type(e).__name__)
    print(e)

200


<html>

    <head><link rel="alternate" type="text/xml" href="/EnterpriseWebService/Service.asmx?disco" />

    <style type="text/css">
    
		BODY { color: #000000; background-color: white; font-family: Verdana; margin-left: 0px; margin-top: 0px; }
		#content { margin-left: 30px; font-size: .70em; padding-bottom: 2em; }
		A:link { color: #336699; font-weight: bold; text-decoration: underline; }
		A:visited { color: #6699cc; font-weight: bold; text-decoration: underline; }
		A:acti


In [3]:
from dotenv import load_dotenv
import os

load_dotenv("/Users/paige.blackstone/idg-analytics-playground/.env")

keys = [
    "ENTERPRISE_WSDL_URL",
    "ENTERPRISE_USER",
    "ENTERPRISE_PASSWORD"]

for key in keys:
    value = os.getenv(key)
    print(f"{key}: {'FOUND' if value else 'MISSING'}")

ENTERPRISE_WSDL_URL: FOUND
ENTERPRISE_USER: FOUND
ENTERPRISE_PASSWORD: FOUND


In [7]:
from dotenv import load_dotenv
import os
import requests
from pathlib import Path

load_dotenv("/Users/paige.blackstone/idg-analytics-playground/.env")

wsdl_url = os.getenv("ENTERPRISE_WSDL_URL")

response = requests.get(wsdl_url, timeout=30)
response.raise_for_status()

wsdl_text = response.text

wsdl_text_clean = wsdl_text.replace(
    "http://localhost/EnterpriseWebService/Enterprise Connect",
    "http://localhost/EnterpriseWebService/EnterpriseConnect"
)

wsdl_dir = Path("/Users/paige.blackstone/idg-analytics-playground/wsdl")
wsdl_dir.mkdir(exist_ok=True)

local_wsdl_path = wsdl_dir / "EnterpriseWebService_clean.wsdl"
local_wsdl_path.write_text(wsdl_text_clean, encoding="utf-8")

print(f"Saved cleaned WSDL to: {local_wsdl_path}")

from zeep import Client

client = Client(wsdl=str(local_wsdl_path))

print("Connected to cleaned local WSDL")

Saved cleaned WSDL to: /Users/paige.blackstone/idg-analytics-playground/wsdl/EnterpriseWebService_clean.wsdl
Connected to cleaned local WSDL


In [8]:
from zeep import Client

client = Client(
    wsdl="/Users/paige.blackstone/idg-analytics-playground/wsdl/EnterpriseWebService_clean.wsdl"
)

print("Connected to cleaned local WSDL")

Connected to cleaned local WSDL


In [9]:
for service in client.wsdl.services.values():
    print(f"\nService: {service.name}")

    for port in service.ports.values():
        print(f"\nPort: {port.name}")

        for operation in sorted(port.binding._operations.keys()):
            print(operation)


Service: EPMS_Connect

Port: EPMS_ConnectSoap
AddComponentToOrder
AddMaterialToDatabase
CXMLSubmitOrder
CompleteLastProductionTransaction
ConvertEstimateToOrder
ConvertToOrder
CreateChangeOrder
CreateInventoryAdjustment
CreateInventoryAdjustmentWithComments
CreateInventoryDeduction
CreateInventoryDeductionWithComments
CreateInventoryTransaction
CreateInventoryTransfer
CreateInventoryTransferWithComments
CreatePOReceipt
CreateReceipt
Four51cXMLSubmitOrder
GetARInvoiceOpenBalance
GetComponentInfo
GetCostCenterList
GetDepartmentList
GetDetailJobStatus
GetDetailJobStatusByOutsideOrderID
GetEmployeeList
GetInventoryStatus
GetInvoiceDetails
GetJobList
GetJobProductionEntries
GetJobSchedule
GetJobShippingStatus
GetOnHandQtyByLocation
GetOrderDetails
GetPurchaseOrder
GetSummaryJobStatus
GetSummaryJobStatusByOutsideOrderID
GetTemplateList
LoadEstimateToConvert
OPSStockLevelLookup
ResetInventoryOnHand
SubmitARInvoice
SubmitARInvoiceBatch
SubmitARInvoiceLineItem
SubmitCBTransaction
SubmitChargeB

In [10]:
print(client.service.GetJobList.__doc__)
print(client.service.GetOrderDetails.__doc__)
print(client.service.GetInvoiceDetails.__doc__)

GetJobList(Credentials: ns0:Authenticator, JobType: xsd:string, FilterType: xsd:string, FilterCriteria: xsd:string, blnPriceOnLineReadyOnly: xsd:boolean, lngNumberOfRecords: xsd:long) -> GetJobListResult: ns0:ArrayOfJobInfo
GetOrderDetails(Credentials: ns0:Authenticator, JobNumber: xsd:string, JobType: xsd:string) -> GetOrderDetailsResult: ns0:OrderInformation
GetInvoiceDetails(Credentials: ns0:Authenticator, TransactionNumber: xsd:string) -> GetInvoiceDetailsResult: ns0:ARTransactionInfo


In [11]:
print(client.service.GetDepartmentList.__doc__)
print(client.service.GetCostCenterList.__doc__)
print(client.service.GetEmployeeList.__doc__)

GetDepartmentList(Credentials: ns0:Authenticator, SendBlanks: xsd:boolean) -> GetDepartmentListResult: ns0:ArrayOfDepartment
GetCostCenterList(Credentials: ns0:Authenticator, SendBlanks: xsd:boolean) -> GetCostCenterListResult: ns0:ArrayOfCostCenter
GetEmployeeList(Credentials: ns0:Authenticator, SendBlanks: xsd:boolean) -> GetEmployeeListResult: ns0:ArrayOfEmployee


In [12]:
client.get_type('ns0:Authenticator')

In [13]:
auth_type = client.get_type('ns0:Authenticator')
print(auth_type)

Authenticator({http://localhost/EnterpriseWebService/EnterpriseConnect}Authenticator(Username: xsd:string, Password: xsd:string))


In [14]:
print(auth_type.elements)

[('Username', <Element(name='Username', type=<zeep.xsd.types.builtins.String object at 0x112dfe650>)>), ('Password', <Element(name='Password', type=<zeep.xsd.types.builtins.String object at 0x112dfe650>)>)]


In [15]:
auth_type = client.get_type('ns0:Authenticator')

credentials = auth_type(
    Username=os.getenv("ENTERPRISE_USER"),
    Password=os.getenv("ENTERPRISE_PASSWORD")
)

In [17]:
from dotenv import load_dotenv
import os
import requests

load_dotenv("/Users/paige.blackstone/idg-analytics-playground/.env")

service_url = "http://128.10.10.3/EnterpriseWebService/Service.asmx"

username = os.getenv("ENTERPRISE_USER")  # or ENTERPRISE_USERNAME, whichever is in your .env
password = os.getenv("ENTERPRISE_PASSWORD")

soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xmlns:xsd="http://www.w3.org/2001/XMLSchema"
               xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetDepartmentList xmlns="http://localhost/EnterpriseWebService/Enterprise Connect">
      <Credentials>
        <Username>{username}</Username>
        <Password>{password}</Password>
      </Credentials>
      <SendBlanks>false</SendBlanks>
    </GetDepartmentList>
  </soap:Body>
</soap:Envelope>"""

headers = {
    "Content-Type": "text/xml; charset=utf-8",
    "SOAPAction": '"http://localhost/EnterpriseWebService/Enterprise Connect/GetDepartmentList"',
}

response = requests.post(
    service_url,
    data=soap_body.encode("utf-8"),
    headers=headers,
    timeout=30
)

print(response.status_code)
print(response.text[:2000])

200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><GetDepartmentListResponse xmlns="http://localhost/EnterpriseWebService/Enterprise Connect"><GetDepartmentListResult /></GetDepartmentListResponse></soap:Body></soap:Envelope>


In [18]:
soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xmlns:xsd="http://www.w3.org/2001/XMLSchema"
               xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetDepartmentList xmlns="http://localhost/EnterpriseWebService/Enterprise Connect">
      <Credentials>
        <Username>{username}</Username>
        <Password>{password}</Password>
      </Credentials>
      <SendBlanks>true</SendBlanks>
    </GetDepartmentList>
  </soap:Body>
</soap:Envelope>"""

response = requests.post(
    service_url,
    data=soap_body.encode("utf-8"),
    headers=headers,
    timeout=30
)

print(response.status_code)
print(response.text[:3000])

200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><GetDepartmentListResponse xmlns="http://localhost/EnterpriseWebService/Enterprise Connect"><GetDepartmentListResult /></GetDepartmentListResponse></soap:Body></soap:Envelope>


In [19]:
soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xmlns:xsd="http://www.w3.org/2001/XMLSchema"
               xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetCostCenterList xmlns="http://localhost/EnterpriseWebService/Enterprise Connect">
      <Credentials>
        <Username>{username}</Username>
        <Password>{password}</Password>
      </Credentials>
      <SendBlanks>true</SendBlanks>
    </GetCostCenterList>
  </soap:Body>
</soap:Envelope>"""

headers = {
    "Content-Type": "text/xml; charset=utf-8",
    "SOAPAction": '"http://localhost/EnterpriseWebService/Enterprise Connect/GetCostCenterList"',
}

response = requests.post(service_url, data=soap_body.encode("utf-8"), headers=headers, timeout=30)

print(response.status_code)
print(response.text[:3000])

200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><GetCostCenterListResponse xmlns="http://localhost/EnterpriseWebService/Enterprise Connect"><GetCostCenterListResult /></GetCostCenterListResponse></soap:Body></soap:Envelope>


In [20]:
headers = {
    "Content-Type": "text/xml; charset=utf-8",
    "SOAPAction": '"http://localhost/EnterpriseWebService/Enterprise Connect/GetJobList"',
}

soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xmlns:xsd="http://www.w3.org/2001/XMLSchema"
               xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetJobList xmlns="http://localhost/EnterpriseWebService/Enterprise Connect">
      <Credentials>
        <Username>{username}</Username>
        <Password>{password}</Password>
      </Credentials>
      <JobType>Order</JobType>
      <FilterType></FilterType>
      <FilterCriteria></FilterCriteria>
      <blnPriceOnLineReadyOnly>false</blnPriceOnLineReadyOnly>
      <lngNumberOfRecords>10</lngNumberOfRecords>
    </GetJobList>
  </soap:Body>
</soap:Envelope>"""

response = requests.post(
    service_url,
    data=soap_body.encode("utf-8"),
    headers=headers,
    timeout=30
)

print(response.status_code)
print(response.text[:5000])

200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><GetJobListResponse xmlns="http://localhost/EnterpriseWebService/Enterprise Connect"><GetJobListResult /></GetJobListResponse></soap:Body></soap:Envelope>


In [21]:
for job_type in ["Order", "Estimate", ""]:
    soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
               xmlns:xsd="http://www.w3.org/2001/XMLSchema"
               xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <GetJobList xmlns="http://localhost/EnterpriseWebService/Enterprise Connect">
      <Credentials>
        <Username>{username}</Username>
        <Password>{password}</Password>
      </Credentials>
      <JobType>{job_type}</JobType>
      <FilterType></FilterType>
      <FilterCriteria></FilterCriteria>
      <blnPriceOnLineReadyOnly>false</blnPriceOnLineReadyOnly>
      <lngNumberOfRecords>25</lngNumberOfRecords>
    </GetJobList>
  </soap:Body>
</soap:Envelope>"""

    headers = {
        "Content-Type": "text/xml; charset=utf-8",
        "SOAPAction": '"http://localhost/EnterpriseWebService/Enterprise Connect/GetJobList"',
    }

    response = requests.post(service_url, data=soap_body.encode("utf-8"), headers=headers, timeout=30)

    print("\nJOB TYPE:", repr(job_type))
    print(response.status_code)
    print(response.text[:2000])


JOB TYPE: 'Order'
200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><GetJobListResponse xmlns="http://localhost/EnterpriseWebService/Enterprise Connect"><GetJobListResult /></GetJobListResponse></soap:Body></soap:Envelope>

JOB TYPE: 'Estimate'
200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema"><soap:Body><GetJobListResponse xmlns="http://localhost/EnterpriseWebService/Enterprise Connect"><GetJobListResult /></GetJobListResponse></soap:Body></soap:Envelope>

JOB TYPE: ''
200
<?xml version="1.0" encoding="utf-8"?><soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.